# Semantic Segmentation Test Metrics

This notebook reads semantic checkpoint-sweep outputs, computes summary metrics, and saves each plot as a PNG under a configurable output directory.


## Config


In [ ]:
from pathlib import Path

# Checkpoint sweep output root. Expected structure:
# BASE_DIR/{graha_model,toy_model}/{last,epoch_000,...}/{sample_id}/*.npy
BASE_DIR = Path("/explore/nobackup/people/ajkerr1/Lunar_FM/full_model_lfm/lfm/notebooks/full_model/outputs/sem_ckpt_sweep_date_2026_07_20-time_09_28_28")

# Checkpoint folder used for the static sample/global/per-chip plots.
CHECKPOINT_NAME = "last"

# Configure plot/table outputs here.
OUTPUT_DIR = Path("/explore/nobackup/people/mfrost2/temp/sem_seg_test_metrics")
PLOT_OUTPUT_DIR = OUTPUT_DIR / "plots"
TABLE_OUTPUT_DIR = OUTPUT_DIR / "tables"
SAVE_PLOTS = True
PLOT_DPI = 200

GRAHA_DIR = BASE_DIR / "graha_model" / CHECKPOINT_NAME
DINO_DIR = BASE_DIR / "toy_model" / CHECKPOINT_NAME
GRAHA_EPOCH_ROOT = BASE_DIR / "graha_model"
DINO_EPOCH_ROOT = BASE_DIR / "toy_model"

COLOR_GT = "#FFC107"
COLOR_GRAHA = "#1E88E5"
COLOR_DINO = "#004D40"
MODEL_PALETTE = {"Graha": COLOR_GRAHA, "DINOv3": COLOR_DINO}

PLOT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TABLE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"BASE_DIR: {BASE_DIR}")
print(f"Checkpoint: {CHECKPOINT_NAME}")
print(f"Plot output: {PLOT_OUTPUT_DIR}")
print(f"Table output: {TABLE_OUTPUT_DIR}")


## Imports And Helpers


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    confusion_matrix,
    jaccard_score,
    precision_score,
    recall_score,
)


def save_figure(fig, filename: str) -> Path:
    """Save a matplotlib figure if SAVE_PLOTS is enabled."""
    save_path = PLOT_OUTPUT_DIR / filename
    if SAVE_PLOTS:
        fig.savefig(save_path, dpi=PLOT_DPI, bbox_inches="tight")
        print(f"Saved plot to {save_path}")
    return save_path


def mask_2d(arr: np.ndarray) -> np.ndarray:
    """Return a 2D mask from possible singleton/channel-first arrays."""
    if arr.ndim == 3 and arr.shape[0] == 1:
        return arr[0]
    if arr.ndim == 3 and arr.shape[-1] == 1:
        return arr[..., 0]
    return arr


def display_band(image: np.ndarray, band: int = 0) -> np.ndarray:
    """Return one displayable image band from C,H,W input arrays."""
    if image.ndim == 3:
        return image[band]
    return image


def create_overlay(mask: np.ndarray, hex_color: str, alpha: float = 0.5) -> np.ndarray:
    """Convert a binary mask to an RGBA overlay."""
    rgb = mcolors.to_rgb(hex_color)
    overlay = np.zeros((*mask.shape, 4), dtype=np.float32)
    overlay[..., :3] = rgb
    overlay[..., 3] = (mask > 0).astype(np.float32) * alpha
    return overlay


def sample_dirs(model_dir: Path) -> list[Path]:
    return sorted([path for path in model_dir.iterdir() if path.is_dir()])


def load_sample_arrays(sample_id: str) -> dict[str, np.ndarray]:
    graha_sample_dir = GRAHA_DIR / sample_id
    dino_sample_dir = DINO_DIR / sample_id
    return {
        "input": np.load(graha_sample_dir / f"{sample_id}_input.npy"),
        "label": mask_2d(np.load(graha_sample_dir / f"{sample_id}_label.npy")),
        "graha_pred": mask_2d(np.load(graha_sample_dir / f"{sample_id}_pred.npy")),
        "dino_pred": mask_2d(np.load(dino_sample_dir / f"{sample_id}_pred.npy")),
    }


def binary_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, average="binary", zero_division=0),
        "Recall": recall_score(y_true, y_pred, average="binary", zero_division=0),
        "IoU": jaccard_score(y_true, y_pred, average="binary", zero_division=0),
    }


## Select Sample


In [ ]:
patch_dirs = [path.name for path in sample_dirs(GRAHA_DIR)]
if not patch_dirs:
    raise FileNotFoundError(f"No sample directories found in {GRAHA_DIR}")

# Change this manually if you want a specific sample.
SAMPLE_CHIP = patch_dirs[0]

print(f"Found {len(patch_dirs)} samples in {GRAHA_DIR}")
print(f"Selected sample: {SAMPLE_CHIP}")


## Plot Sample Overlays


In [ ]:
arrays = load_sample_arrays(SAMPLE_CHIP)
img_to_plot = display_band(arrays["input"], band=0)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle(f"Visualizing Chip: {SAMPLE_CHIP}", fontsize=16, fontweight="bold")

plot_specs = [
    ("Ground Truth Label", arrays["label"], COLOR_GT),
    ("Graha Prediction", arrays["graha_pred"], COLOR_GRAHA),
    ("DINOv3 Prediction", arrays["dino_pred"], COLOR_DINO),
]

for ax, (title, mask, color) in zip(axes, plot_specs):
    ax.imshow(img_to_plot, cmap="gray")
    ax.imshow(create_overlay(mask, color, alpha=0.5), interpolation="none")
    ax.set_title(title, color=color, fontweight="bold")
    ax.axis("off")
    ax.legend(handles=[mpatches.Patch(color=color, alpha=0.5, label="Crater")], loc="upper right")

plt.tight_layout()
save_figure(fig, f"sample_overlay_{SAMPLE_CHIP}.png")
plt.show()


## Inspect Sample Metrics Files


In [ ]:
def inspect_metrics(model_name: str, chip_dir: Path) -> None:
    print(f"=== {model_name} Metrics for {SAMPLE_CHIP} ===")
    txt_path = chip_dir / "metrics.txt"
    npy_path = chip_dir / "metrics.npy"

    if txt_path.exists():
        print("--- metrics.txt ---")
        print(txt_path.read_text().strip())
    else:
        print("metrics.txt not found.")

    print()
    if npy_path.exists():
        print("--- metrics.npy ---")
        data = np.load(npy_path, allow_pickle=True)
        print(data.item() if data.ndim == 0 else data)
    else:
        print("metrics.npy not found.")
    print("\n" + "=" * 50 + "\n")

inspect_metrics("Graha", GRAHA_DIR / SAMPLE_CHIP)
inspect_metrics("DINOv3", DINO_DIR / SAMPLE_CHIP)


## Calculate Per-Chip Metrics For Selected Checkpoint


In [ ]:
metrics_data = []
label_arrays = []
graha_pred_arrays = []
dino_pred_arrays = []

print(f"Processing {len(patch_dirs)} chips from checkpoint '{CHECKPOINT_NAME}'...")

for chip in patch_dirs:
    label_path = GRAHA_DIR / chip / f"{chip}_label.npy"
    graha_pred_path = GRAHA_DIR / chip / f"{chip}_pred.npy"
    dino_pred_path = DINO_DIR / chip / f"{chip}_pred.npy"

    if not (label_path.exists() and graha_pred_path.exists() and dino_pred_path.exists()):
        continue

    label = mask_2d(np.load(label_path)).reshape(-1)
    graha_pred = mask_2d(np.load(graha_pred_path)).reshape(-1)
    dino_pred = mask_2d(np.load(dino_pred_path)).reshape(-1)

    label_arrays.append(label)
    graha_pred_arrays.append(graha_pred)
    dino_pred_arrays.append(dino_pred)

    metrics_data.append({"Model": "Graha", "Chip": chip, **binary_metrics(label, graha_pred)})
    metrics_data.append({"Model": "DINOv3", "Chip": chip, **binary_metrics(label, dino_pred)})

df_metrics = pd.DataFrame(metrics_data)
all_labels = np.concatenate(label_arrays) if label_arrays else np.array([], dtype=np.int64)
all_graha_preds = np.concatenate(graha_pred_arrays) if graha_pred_arrays else np.array([], dtype=np.int64)
all_dino_preds = np.concatenate(dino_pred_arrays) if dino_pred_arrays else np.array([], dtype=np.int64)

print(f"Processed {len(df_metrics) // 2} matched chips.")
df_metrics.head()


## Global Metrics


In [ ]:
global_rows = []
for model_name, preds in [("Graha", all_graha_preds), ("DINOv3", all_dino_preds)]:
    global_rows.append({"Model": model_name, **binary_metrics(all_labels, preds)})

df_global_metrics = pd.DataFrame(global_rows)
df_global_metrics.to_csv(TABLE_OUTPUT_DIR / f"global_metrics_{CHECKPOINT_NAME}.csv", index=False)
print(df_global_metrics.to_string(index=False))
print(f"Saved global metrics to {TABLE_OUTPUT_DIR / f'global_metrics_{CHECKPOINT_NAME}.csv'}")


## Plot Confusion Matrices


In [ ]:
class_names = ["No Crater", "Crater"]
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

cm_graha = confusion_matrix(all_labels, all_graha_preds, normalize="all")
ConfusionMatrixDisplay(cm_graha, display_labels=class_names).plot(
    ax=axes[0], cmap="Blues", colorbar=False, values_format=".1%"
)
axes[0].set_title("Graha Model (% of Total Data)")

cm_dino = confusion_matrix(all_labels, all_dino_preds, normalize="all")
ConfusionMatrixDisplay(cm_dino, display_labels=class_names).plot(
    ax=axes[1], cmap="Greens", colorbar=False, values_format=".1%"
)
axes[1].set_title("DINOv3 Model (% of Total Data)")

plt.tight_layout()
save_figure(fig, f"confusion_matrices_{CHECKPOINT_NAME}.png")
plt.show()


## Plot Per-Chip Metric Distributions


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()
metrics_to_plot = ["Accuracy", "Precision", "Recall", "IoU"]

for i, metric in enumerate(metrics_to_plot):
    sns.boxplot(
        data=df_metrics,
        x="Model",
        y=metric,
        hue="Model",
        legend=False,
        ax=axes[i],
        palette=MODEL_PALETTE,
    )
    axes[i].set_title(f"{metric} Distribution per Chip")
    axes[i].set_xlabel("")
    axes[i].set_ylabel(metric)
    axes[i].yaxis.grid(True, linestyle="--", alpha=0.7)

plt.suptitle("Model Performance Comparison (Per Chip)", fontsize=16, y=1.02)
plt.tight_layout()
save_figure(fig, f"per_chip_metric_distributions_{CHECKPOINT_NAME}.png")
plt.show()


## Calculate Metrics Across Epochs


In [ ]:
epoch_folders = sorted([path.name for path in GRAHA_EPOCH_ROOT.iterdir() if path.is_dir() and path.name.startswith("epoch_")])
print(f"Starting metric calculations across {len(epoch_folders)} epochs...")

graha_history = {"Epoch": [], "Accuracy": [], "Precision": [], "Recall": [], "IoU": []}
dino_history = {"Epoch": [], "Accuracy": [], "Precision": [], "Recall": [], "IoU": []}

for epoch in epoch_folders:
    epoch_num = int(epoch.split("_")[1])
    graha_epoch_dir = GRAHA_EPOCH_ROOT / epoch
    dino_epoch_dir = DINO_EPOCH_ROOT / epoch
    chips = [path.name for path in sample_dirs(graha_epoch_dir)]

    graha_labels, graha_preds = [], []
    dino_labels, dino_preds = [], []

    for chip in chips:
        g_label_path = graha_epoch_dir / chip / f"{chip}_label.npy"
        g_pred_path = graha_epoch_dir / chip / f"{chip}_pred.npy"
        d_label_path = dino_epoch_dir / chip / f"{chip}_label.npy"
        d_pred_path = dino_epoch_dir / chip / f"{chip}_pred.npy"

        if g_label_path.exists() and g_pred_path.exists():
            graha_labels.append(mask_2d(np.load(g_label_path)).reshape(-1))
            graha_preds.append(mask_2d(np.load(g_pred_path)).reshape(-1))

        if d_label_path.exists() and d_pred_path.exists():
            dino_labels.append(mask_2d(np.load(d_label_path)).reshape(-1))
            dino_preds.append(mask_2d(np.load(d_pred_path)).reshape(-1))

    if graha_labels:
        y_true = np.concatenate(graha_labels)
        y_pred = np.concatenate(graha_preds)
        metrics = binary_metrics(y_true, y_pred)
        graha_history["Epoch"].append(epoch_num)
        for metric_name, value in metrics.items():
            graha_history[metric_name].append(value)

    if dino_labels:
        y_true = np.concatenate(dino_labels)
        y_pred = np.concatenate(dino_preds)
        metrics = binary_metrics(y_true, y_pred)
        dino_history["Epoch"].append(epoch_num)
        for metric_name, value in metrics.items():
            dino_history[metric_name].append(value)

    if epoch_num % 10 == 0:
        print(f"Processed through {epoch}...")

df_graha = pd.DataFrame(graha_history)
df_dino = pd.DataFrame(dino_history)
print("Epoch metric calculations complete.")
print(f"Graha epochs: {len(df_graha)} | DINOv3 epochs: {len(df_dino)}")


## Save Epoch Metrics Tables


In [ ]:
graha_csv = TABLE_OUTPUT_DIR / "graha_calculated_metrics_history.csv"
dino_csv = TABLE_OUTPUT_DIR / "dinov3_calculated_metrics_history.csv"

df_graha.to_csv(graha_csv, index=False)
df_dino.to_csv(dino_csv, index=False)

print(f"Saved Graha metrics to {graha_csv}")
print(f"Saved DINOv3 metrics to {dino_csv}")


## Plot Metrics Across Epochs


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
metrics_to_plot = ["Accuracy", "Precision", "Recall", "IoU"]

for i, metric in enumerate(metrics_to_plot):
    ax = axes[i]

    if metric in df_graha.columns and not df_graha.empty:
        ax.plot(df_graha["Epoch"], df_graha[metric], label="Graha", color=COLOR_GRAHA, linewidth=2.5)

    if metric in df_dino.columns and not df_dino.empty:
        ax.plot(df_dino["Epoch"], df_dino[metric], label="DINOv3", color=COLOR_DINO, linewidth=2.5)

    ax.set_title(f"{metric} Progression", fontweight="bold")
    ax.set_xlabel("Epoch")
    ax.set_ylabel(metric)
    ax.grid(True, linestyle="--", alpha=0.7)
    ax.legend(loc="lower right")

plt.suptitle("Model Performance Over Epochs", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
save_figure(fig, "metric_progression_by_epoch.png")
plt.show()
